# cloudflare-ollama-bridge — free remote LLM server on Colab

Runs **Ollama** on Google Colab's free T4 GPU and exposes it to the internet with a **Cloudflare Tunnel**, so you can call any Ollama model from your own machine without owning a GPU.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then run the cells in order.

Set `MODEL_NAME` below to any model from the [Ollama library](https://ollama.com/library) (`qwen3:8b`, `llama3.1:8b`, `mistral`, `gemma2`, ...).

In [ ]:
MODEL_NAME = "qwen3:8b"  # change to any Ollama model tag you want

In [ ]:
# Confirm the GPU is actually attached
!nvidia-smi

In [ ]:
# Ollama's install script needs zstd, which Colab's base image doesn't ship
!apt-get -qq update && apt-get -qq install -y zstd

!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start the Ollama server in the background
import os, subprocess, time

def start_ollama():
    # Kill anything left over from a previous run of this cell.
    # `ollama serve` spawns a `llama-server` child process; killing the
    # parent alone leaves that child orphaned, still holding GPU memory,
    # and the next `ollama serve` will hang on every request until it
    # times out. Kill both explicitly.
    subprocess.run(["pkill", "-f", "ollama serve"])
    subprocess.run(["pkill", "-f", "llama-server"])
    time.sleep(3)

    # Requests coming through the tunnel carry the tunnel's hostname in the
    # Host header. Ollama checks that header against an allowlist (DNS-rebinding
    # protection) and rejects anything that isn't localhost/127.0.0.1 with a
    # 403, so we relax it with OLLAMA_ORIGINS. OLLAMA_HOST=0.0.0.0 is needed so
    # the server accepts connections from outside its own loopback interface.
    env = os.environ.copy()
    env["OLLAMA_HOST"] = "0.0.0.0:11434"
    env["OLLAMA_ORIGINS"] = "*"

    proc = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    time.sleep(5)
    return proc

ollama_proc = start_ollama()
print("ollama serve started, pid:", ollama_proc.pid)

!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
!ollama ps

In [ ]:
# Pull the model (can take a few minutes depending on size and Colab's bandwidth)
!ollama pull {MODEL_NAME}

In [ ]:
# Quick sanity check from inside Colab
import json, subprocess

result = subprocess.run(
    ["curl", "-s", "http://localhost:11434/api/generate", "-d",
     json.dumps({"model": MODEL_NAME, "prompt": "hello, are you working?", "stream": False})],
    capture_output=True, text=True,
)
print(result.stdout)

In [ ]:
# Expose port 11434 to the internet with a Cloudflare quick tunnel
import subprocess, time, re

def start_tunnel():
    # Kill a leftover tunnel from a previous run (otherwise the download
    # below fails with "Text file busy").
    subprocess.run(["pkill", "-f", "cloudflared"])
    time.sleep(2)

    proc = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://localhost:11434"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    url = None
    for _ in range(60):
        line = proc.stdout.readline()
        if not line:
            continue
        print(line.strip())
        match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            break
    return proc, url

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

tunnel_proc, public_url = start_tunnel()

print("\n=== PUBLIC URL ===")
print(public_url)
print(f"\nPut this in your local .env as MODEL_SERVER_URL, and MODEL_NAME={MODEL_NAME}")

## Step 6 (optional) — keep-alive watchdog

Two things cause this setup to silently stop working between the moments you're actively looking at it:

1. **Idle disconnect.** Colab's free tier disconnects a runtime after roughly 90 minutes with no activity in the tab/kernel.
2. **Model unload.** Ollama drops a model from GPU memory after ~5 minutes unused — the next request then pays a slow "cold start" reload.
3. **The orphaned-process bug from Step 3** can also happen on its own if `ollama serve` or `cloudflared` crash mid-session.

The cell below starts a background thread that, every 60 seconds: pings the model to keep it warm (also counts as runtime activity, which helps against idle disconnect), and restarts `ollama serve`/`cloudflared` if either process has died.

**This is a best-effort mitigation, not a guarantee** — it cannot survive you closing the Colab tab, running past the hard ~12h session cap, or Colab reclaiming the GPU under heavy platform load.

In [ ]:
import threading, time, json, subprocess, requests

WATCHDOG_INTERVAL_S = 60
_watchdog_stop = threading.Event()

def _watchdog():
    global ollama_proc, tunnel_proc, public_url
    while not _watchdog_stop.is_set():
        _watchdog_stop.wait(WATCHDOG_INTERVAL_S)
        if _watchdog_stop.is_set():
            break

        if ollama_proc.poll() is not None:
            print(f"[watchdog {time.strftime('%H:%M:%S')}] ollama serve died — restarting")
            ollama_proc = start_ollama()

        if tunnel_proc.poll() is not None:
            print(f"[watchdog {time.strftime('%H:%M:%S')}] cloudflared died — restarting (new URL!)")
            tunnel_proc, public_url = start_tunnel()
            print(f"[watchdog {time.strftime('%H:%M:%S')}] new public URL: {public_url}")

        try:
            resp = requests.post(
                "http://localhost:11434/api/generate",
                json={"model": MODEL_NAME, "prompt": "ping", "stream": False, "keep_alive": "10m"},
                timeout=30,
            )
            ok = resp.status_code == 200
        except Exception as exc:
            ok = False
            print(f"[watchdog {time.strftime('%H:%M:%S')}] heartbeat request failed: {exc}")
        if ok:
            print(f"[watchdog {time.strftime('%H:%M:%S')}] heartbeat ok, model kept warm")

_watchdog_thread = threading.Thread(target=_watchdog, daemon=True)
_watchdog_thread.start()
print(f"Watchdog running every {WATCHDOG_INTERVAL_S}s. Run `_watchdog_stop.set()` in a new cell to stop it.")

## Notes

- `public_url` is different every time the tunnel (re)starts — Cloudflare's free quick tunnel is not a stable address. If the watchdog restarts `cloudflared`, check its printed output for the new URL and update your local `.env`.
- If the Colab session disconnects entirely (closed tab, used-up free quota, hard session cap), the server and tunnel both die — rerun the notebook from the Ollama-start cell onward.
- If requests start timing out after ~120s with `Error 524`, you likely have an orphaned `llama-server` process — rerun the "start the Ollama server" cell above (it kills both processes), or fully restart the runtime (`Runtime → Restart session`).
- From your own machine, test the connection with:
  ```bash
  uvicorn demo.app:app --reload --port 8000
  # or: docker compose up
  ```
  See the repo README for the local demo chat UI.